# Imports

In [ ]:
import traceback
from src.equation_discovery.config_equations_for_each_dataset import ConfigEquationDiscovery
from src.equation_discovery.evaluate_equation import test_equation, map_equation_to_syntax_tree, evaluate_equation
from src.analyse_equations.collect_best_equations import load_proposed_equations, add_proposed_equations, save_proposed_equation, system_data, all_data, get_train_test_files, proposed_equation_to_df, \
    add_units, plot_error_per_system, histogram_for_features, abs_difference_between_equation
from src.preprocess_data.preprocess_data import prepare_dataset, split_train_test, get_unit_dict
from src.analyse_equations.config_collect_best_equation import ConfigPlotBestEquation
from src.preprocess_data.config_load_dataset import ConfigLoadData
from SyntaxTree.src.syntax_tree.config_syntax_tree import ConfigSyntaxTree
from src.utils.config_hyperparameter import ConfigHyperparameter
import pandas as pd

# Configs

In [ ]:
parser =ConfigHyperparameter.arguments_parser()
parser= ConfigLoadData.arguments_parser(parser)
parser = ConfigEquationDiscovery.arguments_parser(parser)
parser = ConfigPlotBestEquation.arguments_parser(parser)
parser = ConfigSyntaxTree.arguments_parser(parser)
args = parser.parse_args()
args.save_path = args.ROOT_DIR /f'results/Xiaomei/equation_set_{args.equation_set_id}.json'
args.unit_dict = get_unit_dict(args)
args.unit_dict['y'] = args.unit_dict[args.target]
args.unit_dimension = 5
args.features= ['drop_length', 'adv', 'rec','avg_vel', 'width']

# Results to load

In [ ]:
proposed_equations = load_proposed_equations(args)
add_proposed_equations(args, proposed_equations)

# Dataset to load 

In [ ]:
files_test, files_train = get_train_test_files(args, proposed_equations)
filtered_dfs_train = prepare_dataset(args, files_train)
filtered_dfs_test = prepare_dataset(args, files_test)

# Calculate Error over all data and per system 

In [ ]:
equation_list = list(proposed_equations.keys())
for equation in equation_list:
    try:
        tree = all_data(args, equation, filtered_dfs_test, filtered_dfs_train, proposed_equations)
        system_data(args, equation, filtered_dfs_test, proposed_equations, tree)
        add_units(args, equation, filtered_dfs_train, proposed_equations, tree)
    except Exception as e:
        del proposed_equations[equation]
        print(traceback.format_exc())
        print(e)

# Save proposed equation 

In [ ]:
save_proposed_equation(args, files_test, files_train, proposed_equations)

# Equations sorted after validation error 

In [ ]:
num_variables = 1
pd.options.display.float_format = '${:.2e}'.format
df = proposed_equation_to_df(proposed_equations, num_variables)
df

# Analyse one equation 

In [ ]:
index = 76
equation=proposed_equations[df.iloc[index].loc['equation']]
tree = map_equation_to_syntax_tree(args, df.iloc[index].loc['equation'], infix=False, catch_exceptions=False)
for id , units in equation['units'].items():
    i = float(id.split('_')[0])
    tree.dict_of_nodes[i].units.units = units
print(tree.rearrange_equation_infix_notation())
tree.print()

In [ ]:
plot_error_per_system(df, index, proposed_equations)

# Analyse Feature 

In [ ]:
pass
histogram_for_features(args, filtered_dfs_test, tree)

# Compare two equations

In [ ]:
index_0 = 76
index_1 = 86
abs_difference_between_equation(args, df, filtered_dfs_test, index_0, index_1)